# Measure tissue thickness

Part of every FOV's z-stack is above the tissue, which wastes imaging time.
This notebook maps, per FOV, where (in z, um) the tissue signal ends, from one
finished round (default: `cells`).

In the `multi_z` pipeline, run it after that pipeline's 01 (cells HAL config)
and 03 (positions), once the cells round is imaged.
`multi_z_margin_export.ipynb` continues from its caches (margin, savings,
z table for multi_z 04). Every heavy step is a cached calculation cell
(`analysis/cache/measure_tissue_thickness/`) followed by a display cell.

**Method:** every downsampled pixel gets its topmost foreground z (an
elevation map); a FOV's depth (`z_last_um`) is its map's maximum. Functions in
`MERci.analysis.elevation`.

Steps:
1. Frame table and `CHANNEL_NM`'s z-planes of the round.
2. Boundary (outer grid) vs. interior FOVs and grid indices (`identify_boundary_fovs`).
3. Flat-field (FFC) field from the interior FOVs' z-projections (default
   `min`). SLURM array option.
4. Background/foreground `THRESHOLD` from the boundary FOVs' mid-z frame.
   Check the histogram; override if wrong.
5. Every FOV, every z: FFC-correct, downsample, threshold, keep each pixel's
   topmost foreground z (`compute_fov_elevation`). The heaviest step; SLURM
   array option.
6. Each FOV's `z_last_um` (`NaN` = no signal) and the stitched heatmap
   (`create_elevation_heatmap`) with its histogram.
7. Mosaic at one depth (`Z_MOSAIC_UM`, default 25 um; `create_z_mosaic`).
8. z-sweep GIF (`create_gif`) and MP4.

## 1 — Setup

In [ ]:
import os
import sys
import json
import csv
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from skimage.transform import resize as sk_resize

MERCI_DIR  = Path(os.getcwd()).parent.parent.parent   # MERci/ (notebook lives in MERci/notebooks/after_imaging/measure_thickness/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.config      import ExperimentConfig
from MERci.common.metadata    import ExperimentMetadata
from MERci.common.experiment_info import resolve_sample_identity, positions_file_tag
from MERci.progress_display   import ProgressReporter, format_duration
from MERci.common.io          import read_image_frames, iter_image_frames
from MERci.acquisition.configs import (
    load_round_frame_table, read_hal_exposure_time, get_fov_geometry, get_color_frame_indices,
)
from MERci.acquisition.configs import load_microscope_orientation, apply_microscope_orientation
from MERci.analysis.elevation import (
    identify_boundary_fovs, compute_fov_projection, calculate_ffc, ffc_correct_and_downsample,
    estimate_background_threshold, compute_fov_elevation, create_elevation_heatmap,
    create_z_mosaic, create_gif, create_movie,
)
from MERci.analysis.ffc        import save_ffc_field, load_ffc_field
from MERci.analysis.fov        import atomic_save
from MERci.analysis.round      import create_mosaic
from MERci.plots.experiment_plots       import display_mosaic, get_merci_figures_dir
from MERci.scheduler           import resolve_round_flip_y

print(f"SAMPLE_DIR : {SAMPLE_DIR}")

## 2 — Parameters

In [ ]:
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(MERCI_DIR)
POSITIONS_TAG = positions_file_tag(SAMPLE_NAME, IMAGING_DIR)
IMAGE_SUFFIX = ".zarr"   # must match what HAL wrote

# Microscope (and objective; None = its default).
MICROSCOPE = "ST2"
OBJECTIVE  = None

# Round: by imaging_type (default "cells"), or set ROUND_ID.
ROUND_IMAGING_TYPE = "cells"
ROUND_ID            = None

# Channel to measure tissue depth for.
CHANNEL_NM = 405.0

# Downsample factor for FFC, heatmap and GIF (2304 / 16 = 144 px).
DOWNSAMPLE_FACTOR = 16

# FFC: per-pixel z-projection pooled over interior FOVs: "min" (recommended), "max", "median", "mean".
FFC_METHOD               = "min"
FFC_SMOOTH_SIGMA_PX      = 50.0   # 0 = no smoothing
FFC_NORMALIZE_PERCENTILE = 99.99
FFC_MIN_VALUE            = 0.10

# THRESHOLD estimate (FFC-corrected, downsampled): the BACKGROUND_PERCENTILE
# value of the N_BACKGROUND_FRAMES darkest boundary FOVs.
N_BACKGROUND_FRAMES   = 20
BACKGROUND_PERCENTILE = 100.0

# None = auto-estimate (section 6). Check its plot on a new experiment.
THRESHOLD = None

# Heatmap color scale upper bound (micron).
MAX_Z_COLORMAP = 70.0

# Mosaic at one depth.
Z_MOSAIC_UM = 25.0

# z-sweep GIF.
GIF_Z_STRIDE          = 1        # every Nth z-plane (1 = every frame)
GIF_FRAME_DURATION_MS = 300
GIF_SCALEBAR_UM       = 1000.0   # scale-bar length (um)
GIF_PERCENTILE_CLIP   = (1.0, 99.0)

# MP4 playback speed; None = 1000 / GIF_FRAME_DURATION_MS.
MOVIE_FPS = None

# Plot font sizes.
PLOT_TITLE_FONTSIZE    = 14
PLOT_LABEL_FONTSIZE    = 12
PLOT_TICK_FONTSIZE     = 11
PLOT_LEGEND_FONTSIZE   = 10
PLOT_SUPTITLE_FONTSIZE = 15

print(f"Sample name        : {SAMPLE_NAME}")
print(f"Positions tag       : {POSITIONS_TAG}")
print(f"Microscope         : {MICROSCOPE}  (objective override: {OBJECTIVE})")
print(f"Round imaging type : {ROUND_IMAGING_TYPE}  (ROUND_ID override: {ROUND_ID})")
print(f"Channel            : {CHANNEL_NM} nm")
print(f"Downsample factor  : {DOWNSAMPLE_FACTOR}")
print(f"FFC method         : {FFC_METHOD}")

In [ ]:
pixel_size_um, image_size_px = get_fov_geometry(MICROSCOPE, OBJECTIVE)

config = ExperimentConfig.from_sample_dir(
    SAMPLE_DIR,
    positions_txt  = SAMPLE_DIR / "positions" / f"positions_{POSITIONS_TAG}.txt",
    image_suffix   = IMAGE_SUFFIX,
    microscope     = MICROSCOPE,
    pixel_size_um  = pixel_size_um,
    image_size_px  = image_size_px,
)

meta = ExperimentMetadata.load(config.round_info_csv, config.positions_txt, config.data_dir,
                                   image_suffix=config.image_suffix)

NOTEBOOK_NAME = "measure_tissue_thickness"
figures_dir = get_merci_figures_dir(SAMPLE_DIR, "after_imaging", NOTEBOOK_NAME)
figures_dir.mkdir(parents=True, exist_ok=True)

# Calculation cells cache under analysis/cache/<notebook_name>/.
cache_dir = config.analysis_dir / "cache" / NOTEBOOK_NAME
cache_dir.mkdir(parents=True, exist_ok=True)

# Microscope parameters (orientation) applied to every raw frame read below.
MICROSCOPE_ORIENTATION_DIR = MERCI_DIR / "data" / "configs" / "merlin" / "microscope"
MICROSCOPE_ORIENTATION = load_microscope_orientation(MICROSCOPE, MICROSCOPE_ORIENTATION_DIR)

print(f"Rounds : {meta.n_rounds}")
print(f"FOVs   : {meta.n_fovs}")
print(f"Step size: {config.step_size_um:.2f} um   image_size_px: {config.image_size_px}")
print(f"Microscope orientation ({MICROSCOPE}): {MICROSCOPE_ORIENTATION}")
print(f"Figures: {figures_dir}")
print(f"Cache  : {cache_dir}")

## 3 — Resolve the target round and its frame table

In [ ]:
target_round_id = ROUND_ID if ROUND_ID is not None else meta.round_for_imaging_type(ROUND_IMAGING_TYPE)
if not meta.round_fully_written(target_round_id):
    print(f"WARNING: round {target_round_id} is not yet fully written on disk -- "
          f"results below will be based on a partial FOV set.")

frame_table = load_round_frame_table(target_round_id, config, meta)
round_info  = meta.rounds[target_round_id]

channel_frames = (
    frame_table[frame_table["color"].round(0) == round(CHANNEL_NM)]
    .sort_values("z")
)
if channel_frames.empty:
    raise ValueError(f"No frames found for channel {CHANNEL_NM} nm in round {target_round_id}'s frame table.")

# [(frame index, z um), ...], ascending z; also as two lists for
# MERci.analysis.elevation.
z_frame_indices = list(zip(channel_frames.index.tolist(), channel_frames["z"].tolist()))
frame_idx_list  = [idx for idx, _ in z_frame_indices]
z_um_list       = [z for _, z in z_frame_indices]
mid_frame_idx   = get_color_frame_indices(frame_table)[CHANNEL_NM]

# {fov_id: (x, y)} of this round's FOVs (no transit FOVs).
positions = {fov_id: meta.fovs[fov_id].position
             for fov_id in round_info.fov_files if round_info.fov_files[fov_id]}

print(f"Target round : {target_round_id}  ({len(positions)} FOV(s) with real files)")
print(f"Channel {CHANNEL_NM} nm has {len(z_frame_indices)} z-step(s) in this round's frame table.")

## 4 — Identify boundary FOVs

Boundary FOVs are the grid's outer FOVs (perimeter and hole edges;
`identify_boundary_fovs`). They are used only for the threshold (section 6).
The FFC field (section 5) uses interior FOVs, since many boundary FOVs have
tissue.

In [ ]:
boundary_fov_ids, interior_fov_ids, grid_indices = identify_boundary_fovs(
    positions, config.step_size_um,
    connectivity=config.ffc_connectivity, tolerance_fraction=config.ffc_neighbor_tolerance,
)
print(f"{len(boundary_fov_ids)} / {len(positions)} FOVs are boundary (exterior-grid) FOVs")
print(f"{len(interior_fov_ids)} / {len(positions)} FOVs are interior FOVs")

In [ ]:
half = config.step_size_um / config.non_overlap_fraction / 2   # FOV half-width

def plot_fov_highlight(highlight_ids, highlight_label, highlight_color, title, figure_name):
    fig, ax = plt.subplots(figsize=(8, 7))
    for fov_id, (x, y) in positions.items():
        is_hl = fov_id in highlight_ids
        ax.add_patch(mpatches.Rectangle(
            (x - half, y - half), 2 * half, 2 * half,
            lw=0.3, edgecolor=highlight_color if is_hl else "0.6",
            facecolor=highlight_color if is_hl else "0.6",
            alpha=0.6 if is_hl else 0.15,
        ))
    ax.plot([], [], "s", color=highlight_color, alpha=0.6, ms=8, label=f"{highlight_label} ({len(highlight_ids)})")
    ax.plot([], [], "s", color="0.6", alpha=0.3, ms=8, label=f"other ({len(positions) - len(highlight_ids)})")
    ax.invert_yaxis(); ax.axis("equal")
    ax.set_title(title, fontsize=PLOT_TITLE_FONTSIZE)
    ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
    ax.legend(fontsize=PLOT_LEGEND_FONTSIZE, loc="best")
    fig.tight_layout()
    fig.savefig(figures_dir / f"{figure_name}_round{target_round_id}.png", dpi=150)
    plt.show()

plot_fov_highlight(boundary_fov_ids, "boundary (exterior-grid)", "tab:red",
                    f"Boundary vs. interior FOVs -- round {target_round_id}", "tissue_thickness_boundary_fovs")

## 5 — FFC field: every interior FOV's own full-z-stack projection

Default `min` projection: a nucleus fills only a few z-planes at any pixel, so
the per-pixel minimum is almost always background. `FFC_METHOD` also takes
`"max"`, `"median"`, `"mean"`.

Reading every interior z-stack serially can take hours: the SLURM array
option runs one task per FOV. Re-run this cell after the job finishes; the
field is built once every FOV is done.

In [ ]:
projections_dir = cache_dir / "fov_projections" / f"round{target_round_id}_{int(CHANNEL_NM)}nm"
projections_dir.mkdir(parents=True, exist_ok=True)

interior_paths = {fov_id: projections_dir / f"fov{fov_id:04d}_{FFC_METHOD}.npy" for fov_id in interior_fov_ids}
to_compute = [f for f in interior_fov_ids if not interior_paths[f].exists()]
print(f"{len(interior_fov_ids) - len(to_compute)} / {len(interior_fov_ids)} interior FOV "
      f"'{FFC_METHOD}' projection(s) already cached; {len(to_compute)} more needed.")

USE_SLURM_ARRAY_FFC     = True   # False = serial, local (slow)
SLURM_ARRAY_CONCURRENCY_FFC = 50
SLURM_MEM_FFC           = "8gb"
SLURM_TIME_FFC          = "00:15:00"

if to_compute and USE_SLURM_ARRAY_FFC:
    from MERci.acquisition.cluster_submit import build_fov_projections_array_script, submit_sbatch, is_job_active

    ffc_job_sentinel = cache_dir / f"fov_projections_job_round{target_round_id}.json"
    cached_job = json.loads(ffc_job_sentinel.read_text()) if ffc_job_sentinel.exists() else None

    if (cached_job is not None and cached_job.get("n_pending") == len(to_compute)
            and is_job_active(cached_job["job_id"])):
        print(f"SLURM array job {cached_job['job_id']} is still active "
              f"({len(to_compute)} FOV(s) pending) -- re-run this cell later once it finishes.")
    else:
        manifest_path = cache_dir / f"fov_projections_manifest_round{target_round_id}.csv"
        with open(manifest_path, "w", newline="") as fh:
            writer = csv.writer(fh)
            writer.writerow(["fov_id", "image_path"])
            for fov_id in to_compute:
                writer.writerow([fov_id, round_info.fov_files[fov_id][0]])

        script_path = cache_dir / f"fov_projections_round{target_round_id}.sh"
        build_fov_projections_array_script(
            sample_dir=SAMPLE_DIR, manifest_path=manifest_path, output_dir=projections_dir,
            frame_indices=frame_idx_list, statistics=[FFC_METHOD], orientation=MICROSCOPE_ORIENTATION,
            n_pending=len(to_compute), output_path=script_path,
            array_concurrency=SLURM_ARRAY_CONCURRENCY_FFC, mem=SLURM_MEM_FFC, time=SLURM_TIME_FFC,
        )
        job_id = submit_sbatch(script_path)
        if job_id is not None:
            ffc_job_sentinel.write_text(json.dumps({"job_id": job_id, "n_pending": len(to_compute)}))
            print(f"Submitted SLURM array job {job_id} for {len(to_compute)} FOV(s) -- "
                  f"re-run this cell later once it finishes to load the results.")
        else:
            print("sbatch submission failed (see the logged error above) -- fix the issue and re-run this cell.")
elif to_compute:
    reporter = ProgressReporter(total=len(to_compute), label=f"Computing interior-FOV '{FFC_METHOD}' projections (local)")
    for fov_id in reporter.wrap(to_compute):
        img = compute_fov_projection(round_info.fov_files[fov_id][0], frame_idx_list, FFC_METHOD,
                                      orientation=MICROSCOPE_ORIENTATION)
        np.save(interior_paths[fov_id], img)

ffc_field_path = cache_dir / f"ffc_field_{FFC_METHOD}_round{target_round_id}_{int(CHANNEL_NM)}nm.npz"
if ffc_field_path.exists():
    ffc_field, ffc_meta = load_ffc_field(ffc_field_path)
    print(f"Loaded cached FFC field: {ffc_field_path}  ({ffc_meta})")
else:
    ffc_field, ffc_meta = calculate_ffc(
        sorted(interior_fov_ids), projections_dir, method=FFC_METHOD,
        smooth_sigma_px=FFC_SMOOTH_SIGMA_PX, normalize_percentile=FFC_NORMALIZE_PERCENTILE,
        ffc_min_value=FFC_MIN_VALUE,
    )
    if ffc_field is None:
        print(f"Still waiting on {len(ffc_meta['missing_fov_ids'])} interior FOV projection(s) -- "
              f"re-run this cell later once the SLURM array job above finishes.")
    else:
        save_ffc_field(ffc_field_path, ffc_field, {**ffc_meta, "round_id": target_round_id, "color": CHANNEL_NM})
        print(f"Computed + saved FFC field: {ffc_field_path}  ({ffc_meta})")

In [ ]:
if ffc_field is None:
    print("FFC field not ready yet (see the cell above) -- skipping.")
else:
    cv = float(ffc_field.std() / ffc_field.mean())
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(ffc_field, cmap="viridis", vmin=0, vmax=1.05)
    ax.set_title(f"FFC field ({FFC_METHOD}, {ffc_meta['n_samples']} interior FOV(s))\n"
                 f"coeff. of variation = {cv:.3f}", fontsize=PLOT_TITLE_FONTSIZE)
    ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
    fig.colorbar(im, ax=ax, shrink=0.8)
    fig.tight_layout()
    fig.savefig(figures_dir / f"tissue_thickness_ffc_field_round{target_round_id}.png", dpi=150)
    plt.show()

## 6 — Background/foreground threshold

`THRESHOLD` = the `BACKGROUND_PERCENTILE` value of the `N_BACKGROUND_FRAMES`
darkest boundary FOVs' mid-z frame, FFC-corrected and downsampled
(`estimate_background_threshold`). Check the histogram. If wrong, set
`THRESHOLD` in section 2 and re-run from here.

In [ ]:
if ffc_field is None:
    print("FFC field not ready yet (see section 5) -- skipping threshold estimation.")
else:
    boundary_ds_path = cache_dir / f"boundary_ds_round{target_round_id}_{int(CHANNEL_NM)}nm.npz"
    if boundary_ds_path.exists():
        _npz = np.load(boundary_ds_path)
        boundary_ds = {int(k.split("_")[1]): _npz[k] for k in _npz.files}
        print(f"Loaded {len(boundary_ds)} cached boundary-FOV downsampled frame(s): {boundary_ds_path}")
    else:
        boundary_ds = {}
        reporter = ProgressReporter(total=len(boundary_fov_ids), label="Reading+correcting boundary FOVs")
        for fov_id in reporter.wrap(sorted(boundary_fov_ids)):
            raw = read_image_frames(round_info.fov_files[fov_id][0], [mid_frame_idx])[0]
            boundary_ds[fov_id] = ffc_correct_and_downsample(
                raw, ffc_field, DOWNSAMPLE_FACTOR, MICROSCOPE_ORIENTATION,
            ).astype(np.float32)
        np.savez_compressed(boundary_ds_path, **{f"fov_{k}": v for k, v in boundary_ds.items()})
        print(f"Saved: {boundary_ds_path}")

    estimated_threshold = estimate_background_threshold(boundary_ds, N_BACKGROUND_FRAMES, BACKGROUND_PERCENTILE)
    print(f"Estimated background noise ceiling (p{BACKGROUND_PERCENTILE:.1f} of the {N_BACKGROUND_FRAMES} "
          f"lowest-mean boundary FOVs, FFC-corrected + downsampled space): {estimated_threshold:.1f}")
    if THRESHOLD is None:
        THRESHOLD = estimated_threshold
    print(f"Using THRESHOLD = {THRESHOLD:.1f}")

In [ ]:
if ffc_field is None:
    print("FFC field not ready yet -- skipping.")
else:
    LOG_BINS = 200
    all_vals = np.concatenate([np.clip(ds, 1, None).ravel() for ds in boundary_ds.values()])
    bin_edges = np.linspace(np.log10(all_vals.min()), np.log10(all_vals.max()), LOG_BINS + 1)

    fig, ax = plt.subplots(figsize=(8, 5))
    pooled_hist = np.zeros(LOG_BINS)
    for fov_id, ds in boundary_ds.items():
        hist, _ = np.histogram(np.log10(np.clip(ds, 1, None)), bins=bin_edges, density=True)
        ax.plot(bin_edges[:-1], hist, "-", lw=0.6, color="0.6", alpha=0.5)
        pooled_hist += hist
    pooled_hist /= len(boundary_ds)
    ax.plot(bin_edges[:-1], pooled_hist, "-", lw=2, color="k", label="pooled (mean over boundary FOVs)")
    ax.axvline(np.log10(max(THRESHOLD, 1)), color="tab:red", ls="--", label=f"THRESHOLD = {THRESHOLD:.0f}")
    ax.set_xlabel("log10(intensity)  [FFC-corrected, downsampled]", fontsize=PLOT_LABEL_FONTSIZE)
    ax.set_ylabel("density", fontsize=PLOT_LABEL_FONTSIZE)
    ax.set_title(f"Boundary-FOV {CHANNEL_NM:.0f} nm intensity overlay ({len(boundary_ds)} FOVs)",
                 fontsize=PLOT_TITLE_FONTSIZE)
    ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
    ax.legend(fontsize=PLOT_LEGEND_FONTSIZE)
    fig.tight_layout()
    fig.savefig(figures_dir / f"tissue_thickness_boundary_histograms_round{target_round_id}.png", dpi=150)
    plt.show()

## 7 — Per-FOV elevation matrices + downsampled z-stacks (full FOV grid)

For every FOV and z-plane: FFC-correct, downsample, threshold, keep each
pixel's topmost foreground z (`compute_fov_elevation`). The downsampled
frames are cached for the mosaic and GIF.

The heaviest step: SLURM array option, one task per FOV. Re-run this cell
after the job finishes.

In [ ]:
elevation_dir = cache_dir / "elevation" / f"round{target_round_id}"
elevation_dir.mkdir(parents=True, exist_ok=True)

def elevation_path(fov_id):
    return elevation_dir / f"fov{fov_id:04d}_elevation.npy"

def stack_path(fov_id):
    return elevation_dir / f"fov{fov_id:04d}_stack.npy"

all_fov_ids = sorted(positions)   # every FOV of this round
to_compute = [f for f in all_fov_ids if not (elevation_path(f).exists() and stack_path(f).exists())]
print(f"{len(all_fov_ids) - len(to_compute)} / {len(all_fov_ids)} FOV(s) already cached; "
      f"{len(to_compute)} more needed ({len(z_frame_indices)} z-plane(s) each).")

USE_SLURM_ARRAY_ELEVATION     = True
SLURM_ARRAY_CONCURRENCY_ELEV  = 50
SLURM_MEM_ELEVATION           = "8gb"
SLURM_TIME_ELEVATION          = "00:15:00"

if ffc_field is None or THRESHOLD is None:
    print("FFC field/THRESHOLD not ready yet (see sections 5-6) -- skipping elevation computation.")
elif to_compute and USE_SLURM_ARRAY_ELEVATION:
    from MERci.acquisition.cluster_submit import build_fov_elevation_array_script, submit_sbatch, is_job_active

    elev_job_sentinel = cache_dir / f"elevation_job_round{target_round_id}.json"
    cached_job = json.loads(elev_job_sentinel.read_text()) if elev_job_sentinel.exists() else None

    if (cached_job is not None and cached_job.get("n_pending") == len(to_compute)
            and is_job_active(cached_job["job_id"])):
        print(f"SLURM array job {cached_job['job_id']} is still active "
              f"({len(to_compute)} FOV(s) pending) -- re-run this cell later once it finishes.")
    else:
        manifest_path = cache_dir / f"elevation_manifest_round{target_round_id}.csv"
        with open(manifest_path, "w", newline="") as fh:
            writer = csv.writer(fh)
            writer.writerow(["fov_id", "image_path"])
            for fov_id in to_compute:
                writer.writerow([fov_id, round_info.fov_files[fov_id][0]])

        script_path = cache_dir / f"elevation_round{target_round_id}.sh"
        build_fov_elevation_array_script(
            sample_dir=SAMPLE_DIR, manifest_path=manifest_path, output_dir=elevation_dir,
            frame_indices=frame_idx_list, z_um_values=z_um_list,
            ffc_field_path=ffc_field_path, threshold=THRESHOLD, downsample_factor=DOWNSAMPLE_FACTOR,
            orientation=MICROSCOPE_ORIENTATION, n_pending=len(to_compute), output_path=script_path,
            array_concurrency=SLURM_ARRAY_CONCURRENCY_ELEV, mem=SLURM_MEM_ELEVATION, time=SLURM_TIME_ELEVATION,
        )
        job_id = submit_sbatch(script_path)
        if job_id is not None:
            elev_job_sentinel.write_text(json.dumps({"job_id": job_id, "n_pending": len(to_compute)}))
            print(f"Submitted SLURM array job {job_id} for {len(to_compute)} FOV(s) -- "
                  f"re-run this cell later once it finishes to load the results.")
        else:
            print("sbatch submission failed (see the logged error above) -- fix the issue and re-run this cell.")
elif to_compute:
    reporter = ProgressReporter(total=len(to_compute), label="Computing per-FOV elevation matrices (local)")
    for fov_id in reporter.wrap(to_compute):
        M, ds_stack = compute_fov_elevation(
            round_info.fov_files[fov_id][0], frame_idx_list, z_um_list,
            ffc_field, THRESHOLD, DOWNSAMPLE_FACTOR, orientation=MICROSCOPE_ORIENTATION,
        )
        np.save(elevation_path(fov_id), M)
        np.save(stack_path(fov_id), ds_stack)

ready_fov_ids = [f for f in all_fov_ids if elevation_path(f).exists() and stack_path(f).exists()]
print(f"{len(ready_fov_ids)} / {len(all_fov_ids)} FOV(s) have both an elevation matrix and a stack ready.")
elevation_matrices = {f: np.load(elevation_path(f)) for f in ready_fov_ids}
stack_paths        = {f: stack_path(f) for f in ready_fov_ids}

In [ ]:
if not elevation_matrices:
    print("No FOV(s) ready yet.")
else:
    example_ids = sorted(elevation_matrices)[:6]
    fig, axes = plt.subplots(1, len(example_ids), figsize=(3 * len(example_ids), 3.2))
    vmax = max(float(elevation_matrices[f].max()) for f in example_ids)
    for ax, fov_id in zip(np.atleast_1d(axes), example_ids):
        im = ax.imshow(elevation_matrices[fov_id], cmap="viridis", vmin=0, vmax=vmax)
        ax.set_title(f"FOV {fov_id}", fontsize=PLOT_TITLE_FONTSIZE)
        ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
    cbar = fig.colorbar(im, ax=np.atleast_1d(axes).tolist(), shrink=0.8, label="elevation (um)")
    cbar.ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
    fig.suptitle("Example per-FOV elevation matrices (pre-crop)", fontsize=PLOT_TITLE_FONTSIZE)
    fig.savefig(figures_dir / f"tissue_thickness_example_fov_elevation_round{target_round_id}.png", dpi=150)
    plt.show()

## 8 — Per-FOV `z_last_um` + tissue thickness heatmap (full FOV grid)

A FOV's depth is its elevation map's maximum (0 -> `NaN`, no signal). The
maps, cropped to their non-overlap area, are stitched into one heatmap
(`create_elevation_heatmap`).

In [ ]:
results_rows = []
for fov_id in ready_fov_ids:
    z_max = float(elevation_matrices[fov_id].max())
    results_rows.append({
        "fov_id":    fov_id,
        "z_last_um": z_max if z_max > 0 else np.nan,
        "x_um":      meta.fovs[fov_id].position[0],
        "y_um":      meta.fovs[fov_id].position[1],
    })
results_df = pd.DataFrame(results_rows)

results_csv = config.analysis_dir / f"tissue_thickness_round{target_round_id}.csv"
results_df.to_csv(results_csv, index=False)
print(f"Derived z_last_um for {len(results_df)} FOV(s); saved to {results_csv}")

In [ ]:
n_no_signal = int(results_df["z_last_um"].isna().sum())
print(f"{n_no_signal} / {len(results_df)} FOV(s) had no downsampled pixel above THRESHOLD at all "
      f"(no detected tissue signal).")
print("z_last_um:")
print(results_df["z_last_um"].describe())

In [ ]:
heatmap = create_elevation_heatmap(elevation_matrices, grid_indices, config, downsample_factor=DOWNSAMPLE_FACTOR)
np.save(cache_dir / f"elevation_heatmap_round{target_round_id}.npy", heatmap)
print(f"Stitched elevation heatmap shape: {heatmap.shape}")

In [ ]:
# Heatmap, colorbar and histogram share the heatmap's height (one axes divider).
from mpl_toolkits.axes_grid1 import make_axes_locatable

# Zero (no-signal) pixels are left out of the histogram and title, not the heatmap.
heatmap_values = heatmap[~np.isnan(heatmap)]
heatmap_values_nonzero = heatmap_values[heatmap_values != 0]

# Full thickness = the round's deepest z-plane. Pixels there have tissue at
# least this deep; the stack just ended.
full_thickness_um = float(heatmap_values_nonzero.max())
pct_full_thickness = 100.0 * np.mean(heatmap_values_nonzero == full_thickness_um)

fig, ax = plt.subplots(figsize=(13, 9))

cmap = plt.cm.turbo.copy(); cmap.set_bad("0.85")
im = ax.imshow(np.ma.masked_invalid(heatmap), cmap=cmap, vmin=0, vmax=MAX_Z_COLORMAP)
ax.set_title(f"Tissue thickness heatmap -- {len(ready_fov_ids)} FOV(s), "
             f"{pct_full_thickness:.1f}% pixels with thickness = {full_thickness_um:.1f} um",
             fontsize=PLOT_TITLE_FONTSIZE)
ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)

divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="3%", pad=0.15)
cbar = fig.colorbar(im, cax=cax, label="thickness (um)")
cbar.ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)

# 1-um bins centred on whole microns (values are on a 0.5 um grid). Log x and
# density, since the full-thickness bin dwarfs the rest.
ax_hist = divider.append_axes("right", size="20%", pad=0.7)
HIST_BIN_EDGES = np.arange(0.5, MAX_Z_COLORMAP + 1.5, 1.0)
ax_hist.hist(heatmap_values_nonzero, bins=HIST_BIN_EDGES, orientation="horizontal",
             color="steelblue", density=True)
ax_hist.set_ylim(0, MAX_Z_COLORMAP)
ax_hist.yaxis.tick_right()
ax_hist.set_xscale("log")
ax_hist.xaxis.tick_top()
ax_hist.xaxis.set_label_position("top")
ax_hist.set_xlabel("density", fontsize=PLOT_LABEL_FONTSIZE)
ax_hist.tick_params(labelsize=PLOT_TICK_FONTSIZE)

fig.suptitle(f"Round {target_round_id} -- tissue depth map", fontsize=PLOT_SUPTITLE_FONTSIZE)
fig.savefig(figures_dir / f"tissue_thickness_heatmap_round{target_round_id}.png", dpi=150)
plt.show()

In [ ]:
# Contour version of the heatmap. extend="max" colours pixels above
# MAX_Z_COLORMAP with the top colour.
fig, ax = plt.subplots(figsize=(11, 9))
ax.set_facecolor("0.85")   # same as the heatmap's masked areas

contour_levels = np.linspace(0, MAX_Z_COLORMAP, 21)
cs = ax.contourf(np.ma.masked_invalid(heatmap), levels=contour_levels,
                  cmap=plt.cm.turbo, vmin=0, vmax=MAX_Z_COLORMAP, extend="max")
ax.invert_yaxis()
ax.set_aspect("equal")
ax.set_title(f"Tissue thickness contour -- round {target_round_id}", fontsize=PLOT_TITLE_FONTSIZE)
ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)

divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="3%", pad=0.15)
cbar = fig.colorbar(cs, cax=cax, label="thickness (um)")
cbar.ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)

fig.savefig(figures_dir / f"tissue_thickness_contour_round{target_round_id}.png", dpi=150)
plt.show()

## 9 — Mosaic at a single z (`Z_MOSAIC_UM`)

One depth (default 25 um) from section 7's cached z-stacks, saved as a PNG,
with the same intensity scale, z label and scale bar as the GIF
(`create_z_mosaic`).

In [ ]:
z_mosaic_path = figures_dir / f"tissue_thickness_z_mosaic_round{target_round_id}.png"
create_z_mosaic(
    stack_paths, z_um_list, grid_indices, config, z_mosaic_path, z_um=Z_MOSAIC_UM,
    downsample_factor=DOWNSAMPLE_FACTOR, scalebar_um=GIF_SCALEBAR_UM, percentile_clip=GIF_PERCENTILE_CLIP,
)
print(f"Saved: {z_mosaic_path}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
ax.imshow(np.array(Image.open(z_mosaic_path).convert("L")), cmap="gray")
ax.set_title(f"Mosaic at z ~ {Z_MOSAIC_UM:.1f} um -- round {target_round_id}", fontsize=PLOT_TITLE_FONTSIZE)
ax.set_xticks([]); ax.set_yticks([])
plt.show()

## 10 — z-sweep GIF

Section 7's cached z-stacks, stitched per z-plane into one GIF (`create_gif`):
one intensity scale for every frame (so fading is real), a z label per frame,
and a scale bar (`GIF_SCALEBAR_UM`).

Frames are cached (`analysis/cache/measure_tissue_thickness/gif_frames/round<N>/`),
so a failed save only redoes the encoding.

In [ ]:
gif_path = figures_dir / f"tissue_elevation_gif_round{target_round_id}.gif"
gif_frame_cache_dir = cache_dir / "gif_frames" / f"round{target_round_id}"
n_cached_frames = len(list(gif_frame_cache_dir.glob("z*.png"))) if gif_frame_cache_dir.exists() else 0
print(f"{n_cached_frames} GIF frame(s) already cached in {gif_frame_cache_dir}")
create_gif(
    stack_paths, z_um_list, grid_indices, config, gif_path,
    downsample_factor=DOWNSAMPLE_FACTOR, z_stride=GIF_Z_STRIDE, frame_duration_ms=GIF_FRAME_DURATION_MS,
    scalebar_um=GIF_SCALEBAR_UM, percentile_clip=GIF_PERCENTILE_CLIP, frame_cache_dir=gif_frame_cache_dir,
)
print(f"Saved: {gif_path}")

In [ ]:
with Image.open(gif_path) as im:
    n_gif_frames = im.n_frames
    first_frame = np.array(im.convert("L"))

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(first_frame, cmap="gray")
ax.set_title(f"First GIF frame ({n_gif_frames} frame(s) total) -- open {gif_path.name} to view the full sweep",
             fontsize=PLOT_TITLE_FONTSIZE)
ax.set_xticks([]); ax.set_yticks([])
plt.show()

## 11 — z-sweep movie (MP4, for PowerPoint)

Section 10's frames as a silent H.264 `.mp4` (`create_movie`), which
PowerPoint inserts as a video ("Insert > Video").

In [ ]:
movie_path = figures_dir / f"tissue_elevation_movie_round{target_round_id}.mp4"
create_movie(
    stack_paths, z_um_list, grid_indices, config, movie_path,
    downsample_factor=DOWNSAMPLE_FACTOR, z_stride=GIF_Z_STRIDE, fps=MOVIE_FPS, frame_duration_ms=GIF_FRAME_DURATION_MS,
    scalebar_um=GIF_SCALEBAR_UM, percentile_clip=GIF_PERCENTILE_CLIP, frame_cache_dir=gif_frame_cache_dir,
)
print(f"Saved: {movie_path}")